# Compose와 pgvector 스택

Docker Desktop의 실습 전용 이미지와 스택만 사용해 결과를 확인합니다.

## 실행 환경 준비

이 노트북 전용 Compose 프로젝트의 기존 자원을 먼저 정리합니다.

## 코드 읽기와 개념 경계

Compose는 여러 컨테이너의 이미지, 네트워크, 볼륨, 환경 값을 하나의 YAML에 선언합니다. 서비스 이름 `db`는 Compose 네트워크 안에서 호스트 이름으로 사용되므로 애플리케이션은 로컬호스트가 아니라 `db`로 PostgreSQL에 연결합니다. `depends_on`과 healthcheck 조건은 시작 순서를 돕지만 데이터베이스가 모든 업무 요청을 받을 준비가 됐는지는 애플리케이션 재시도와 readiness로 다시 확인합니다.

`docker compose down`은 컨테이너를 내리지만 기본적으로 이름 있는 볼륨의 데이터는 남깁니다. `--volumes`를 붙이면 실습 데이터도 제거됩니다. 개발 환경의 편리한 초기화와 운영 데이터 보존은 요구가 반대이므로 같은 종료 명령을 그대로 사용하지 않습니다.

In [1]:
# Docker 명령을 일정한 폴더에서 실행하고 실패도 결과로 받습니다.
from pathlib import Path
import json, subprocess, time, urllib.request
# 챕터 폴더와 notebooks 폴더 어디에서 시작해도 제공 자료를 찾습니다.
candidates = [parent / "labs/part7/ch32-docker" for parent in (Path.cwd(), *Path.cwd().parents)]
candidates += [Path.cwd(), Path.cwd().parent]
LAB = next(path for path in candidates if (path / "Dockerfile.baseline").is_file())
def run(*args: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    return subprocess.run(args, cwd=LAB, text=True, capture_output=True, check=check)
version = run("docker", "version", "--format", "{{.Server.Version}}")
print("Docker Engine", version.stdout.strip())
assert version.stdout.strip()
run("docker", "compose", "-p", "ch32-course-stack", "down", "--volumes", check=False)

Docker Engine 29.4.1


CompletedProcess(args=('docker', 'compose', '-p', 'ch32-course-stack', 'down', '--volumes'), returncode=0, stdout='', stderr='time="2026-09-11T11:06:55+09:00" level=warning msg="Warning: No resource found to remove for project \\"ch32-course-stack\\"."\n')

## 설정 검증

Compose 파일을 정규화해 app과 db 서비스가 있는지 확인합니다.

In [2]:
# 컨테이너 생성 전에 구문과 서비스 이름을 검사합니다.
config = run("docker", "compose", "-p", "ch32-course-stack", "config", "--format", "json")
document = json.loads(config.stdout)
print("services", sorted(document["services"]))
assert set(document["services"]) == {"app", "db"}

services ['app', 'db']


## 실패 재현

스택이 없을 때 호스트의 API 주소는 연결을 거부합니다.

In [3]:
# 서비스 시작 전 연결 실패를 실제로 확인합니다.
try:
    urllib.request.urlopen("http://127.0.0.1:18332/ready", timeout=1)
    before = "unexpected success"
except OSError as exc:
    before = type(exc).__name__
print("before", before)
assert before != "unexpected success"

before URLError


## 수정

DB healthcheck가 통과한 뒤 app을 시작하도록 Compose 스택을 실행합니다.

In [4]:
# --wait는 두 서비스가 healthy가 될 때까지 기다립니다.
run("docker", "compose", "-p", "ch32-course-stack", "up", "--build", "--wait")
ready = urllib.request.urlopen("http://127.0.0.1:18332/ready").read().decode()
count = urllib.request.urlopen("http://127.0.0.1:18332/vector-count").read().decode()
print(ready, count)
assert '"ready"' in ready and '"count":1' in count

{"status":"ready"} {"count":1}


## pgvector와 인덱스 확인

데이터베이스 안에서 확장 버전과 HNSW 인덱스를 직접 조회합니다.

In [5]:
# psql 출력으로 확장과 벡터 인덱스 생성을 검증합니다.
sql = "SELECT extversion FROM pg_extension WHERE extname='vector'; SELECT indexname FROM pg_indexes WHERE indexname LIKE '%hnsw';"
db = run("docker", "compose", "-p", "ch32-course-stack", "exec", "-T", "db", "psql", "-U", "course", "-d", "course", "-Atc", sql)
print(db.stdout.strip())
assert "0.8.6" in db.stdout and "hnsw" in db.stdout
run("docker", "compose", "-p", "ch32-course-stack", "down", "--volumes")

0.8.6
course_vectors_embedding_hnsw


CompletedProcess(args=('docker', 'compose', '-p', 'ch32-course-stack', 'down', '--volumes'), returncode=0, stdout='', stderr=' Container ch32-course-stack-app-1 Stopping \n Container ch32-course-stack-app-1 Stopped \n Container ch32-course-stack-app-1 Removing \n Container ch32-course-stack-app-1 Removed \n Container ch32-course-stack-db-1 Stopping \n Container ch32-course-stack-db-1 Stopped \n Container ch32-course-stack-db-1 Removing \n Container ch32-course-stack-db-1 Removed \n Volume ch32-course-stack_pgdata Removing \n Network ch32-course-stack_default Removing \n Volume ch32-course-stack_pgdata Removed \n Network ch32-course-stack_default Removed \n')